In [1]:
import os

In [2]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [5]:
!pip install wandb tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 15.8 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: protobuf━━━━━━━━━━━━━━━━ 0/3 [sentry-sdk]
    Found existing installation: protobuf 6.33.6 0/3 [sentry-sdk]
    Uninstalling protobuf-6.33.6:━━━━━━━━━━━ 0/3 [sentry-sdk]
      Successfully uninstalled protobuf-6.33.60m 0/3 [sentry-sdk]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [wandb]32m2/3 [wandb]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.80.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
streamlit 1.29.0 requires packaging<24,>=16.8, but you have packaging 26.0 which is incompatible.
streamlit 1.29.0 requires protobuf<5,>=3.20, but you have protobuf 5.29.6 which is incompatible.


In [6]:
import wandb
import numpy as np
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow import keras as k

In [7]:
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/saicharanreddy/.netrc
wandb: Currently logged in as: saicherry006 (saicherry006-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiment Overview

This lab builds on the baseline CNN from Lab1 and explores several modifications:

1. **Deeper architecture** — added a second Conv2D block with more filters
2. **Optimizer swap** — switched from SGD to Adam for faster convergence
3. **Learning rate schedule** — added ExponentialDecay to avoid overshooting later epochs
4. **Higher dropout** — increased from 0.2 to 0.3 to reduce overfitting
5. **Full dataset** — increased sample size from 10k to 60k
6. **More epochs** — trained for 10 epochs instead of 5

All runs are tracked in wandb for easy comparison.

In [9]:
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
import numpy as np
from tensorflow.keras.optimizers.schedules import ExponentialDecay


# --- custom callbacks ---------------------------------------------------------

class LogLRCallback(k.callbacks.Callback):
    """Log optimizer learning rate each epoch."""
    def on_epoch_end(self, epoch, logs=None):
        opt = self.model.optimizer
        lr = opt.learning_rate
        # Handle both fixed LR and LR schedules
        if hasattr(lr, '__call__'):
            lr_val = float(lr(self.model.optimizer.iterations).numpy())
        else:
            lr_val = float(lr.numpy() if hasattr(lr, "numpy") else lr)
        wandb.log({"lr": lr_val}, step=self.model.optimizer.iterations.numpy())

class LogSamplesCallback(k.callbacks.Callback):
    """Log a small table of predictions + images every epoch."""
    def __init__(self, x, y, labels, max_rows=32):
        super().__init__()
        self.x = x[:max_rows]
        self.y = y[:max_rows]
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x, verbose=0)
        y_true = np.argmax(self.y, axis=1)
        y_pred = np.argmax(preds, axis=1)

        table = wandb.Table(columns=["image", "y_true", "y_pred", "correct", "p(y_pred)"])
        for i in range(len(self.x)):
            img = self.x[i].squeeze()
            table.add_data(
                wandb.Image(img),
                self.labels[y_true[i]],
                self.labels[y_pred[i]],
                bool(y_true[i] == y_pred[i]),
                float(np.max(preds[i])),
            )
        wandb.log({f"samples/epoch_{epoch+1}": table})

class ConfusionMatrixCallback(k.callbacks.Callback):
    """Log a confusion matrix from the full validation set each epoch."""
    def __init__(self, x_val, y_val, labels):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, verbose=0)
        y_true = np.argmax(self.y_val, axis=1)
        y_pred = np.argmax(preds, axis=1)
        cm_plot = wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.labels,
        )
        wandb.log({"confusion_matrix": cm_plot})


# --- trainer -----------------------------------------------------------------

class FashionMNISTTrainer:
    def __init__(self, project_name="Lab2-deeper-cnn", run_name="adam_deeper_decay"):
        self.cfg = dict(
            dropout=0.3,           # increased from 0.2 to reduce overfitting
            layer_1_size=64,       # doubled from 32 for richer feature maps
            layer_2_size=128,      # NEW: second conv block
            learn_rate=0.001,      # Adam default; lower than SGD
            decay_steps=1000,      # NEW: LR schedule decay
            decay_rate=0.9,        # NEW: gentle exponential decay
            epochs=10,             # doubled from 5
            batch_size=64,
            sample=60000,          # full dataset instead of 10k
            optimizer="adam",      # NEW: switched from SGD
            dense_units=256,       # NEW: hidden dense layer
        )
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=self.cfg,
            settings=wandb.Settings(start_method="thread"),
        )
        self.config = wandb.config
        self.labels = ["T-shirt/top","Trouser","Pullover","Dress","Coat",
                       "Sandal","Shirt","Sneaker","Bag","Ankle boot"]
        self._prepare_data()

    def _prepare_data(self):
        (xtr, ytr), (xte, yte) = fashion_mnist.load_data()
        n = self.config.sample
        xtr = xtr[:n].astype("float32") / 255.0
        ytr = ytr[:n]
        # use full test set for validation regardless of sample size
        xte = xte.astype("float32") / 255.0
        self.X_train = xtr[..., None]
        self.X_test  = xte[..., None]
        self.y_train = to_categorical(ytr)
        self.y_test  = to_categorical(yte)
        self.num_classes = self.y_test.shape[1]

    def _build_model(self):
        inputs = k.Input(shape=(28, 28, 1))

        # --- Block 1 ---
        x = k.layers.Conv2D(self.config.layer_1_size, (3, 3), activation="relu", padding="same")(inputs)
        x = k.layers.BatchNormalization()(x)          # NEW: stabilises training
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)

        # --- Block 2 (NEW) ---
        x = k.layers.Conv2D(self.config.layer_2_size, (3, 3), activation="relu", padding="same")(x)
        x = k.layers.BatchNormalization()(x)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)

        # --- Classifier head ---
        x = k.layers.Flatten()(x)
        x = k.layers.Dense(self.config.dense_units, activation="relu")(x)  # NEW hidden layer
        x = k.layers.Dropout(self.config.dropout)(x)
        outputs = k.layers.Dense(self.num_classes, activation="softmax")(x)

        model = k.Model(inputs, outputs)

        # --- Optimizer with LR decay schedule (NEW) ---
        lr_schedule = ExponentialDecay(
            initial_learning_rate=self.config.learn_rate,
            decay_steps=self.config.decay_steps,
            decay_rate=self.config.decay_rate,
            staircase=False,
        )
        opt = k.optimizers.Adam(learning_rate=lr_schedule)

        model.compile(
            loss="categorical_crossentropy",
            optimizer=opt,
            metrics=["accuracy"],
        )
        return model

    def _log_model_artifact(self, model):
        summary_lines = []
        model.summary(print_fn=summary_lines.append)
        summary_txt = "\n".join(summary_lines)
        os.makedirs("artifacts", exist_ok=True)
        with open("artifacts/model_summary.txt", "w") as f:
            f.write(summary_txt)

        model_path = "artifacts/model.keras"
        model.save(model_path)

        art = wandb.Artifact("fashion_mnist_model_v2", type="model")
        art.add_file("artifacts/model_summary.txt")
        art.add_file(model_path)
        self.run.log_artifact(art)

    def train(self):
        model = self._build_model()
        model.summary()

        callbacks = [
            WandbMetricsLogger(log_freq=10),
            WandbModelCheckpoint("checkpoints/model-{epoch:02d}.keras", save_weights_only=False),
            LogLRCallback(),
            LogSamplesCallback(self.X_test, self.y_test, self.labels, max_rows=32),
            ConfusionMatrixCallback(self.X_test, self.y_test, self.labels),
            # NEW: stop early if val_loss stops improving
            k.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ]

        model.fit(
            self.X_train, self.y_train,
            validation_data=(self.X_test, self.y_test),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=callbacks,
            verbose=1,
        )

        loss, acc = model.evaluate(self.X_test, self.y_test, verbose=0)
        wandb.log({"final/loss": loss, "final/accuracy": acc})
        print(f"\nFinal Test Accuracy: {acc:.4f} | Loss: {loss:.4f}")

        self._log_model_artifact(model)
        self.run.finish()


FashionMNISTTrainer().train()


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 14, 14, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     1,605,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,683,722 (6.42 MB)

 Trainable params: 1,683,338 (6.42 MB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 47s 50ms/step - accuracy: 0.7707 - loss: 0.7885 - val_accuracy: 0.8716 - val_loss: 0.3514
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 44s 47ms/step - accuracy: 0.8716 - loss: 0.3499 - val_accuracy: 0.8782 - val_loss: 0.3536
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 44s 47ms/step - accuracy: 0.8948 - loss: 0.2878 - val_accuracy: 0.9029 - val_loss: 0.2729
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 45s 48ms/step - accuracy: 0.9040 - loss: 0.2615 - val_accuracy: 0.8986 - val_loss: 0.2909
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 56s 60ms/step - accuracy: 0.9103 - loss: 0.2370 - val_accuracy: 0.8967 - val_loss: 0.3158
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 59s 63ms/step - accuracy: 0.9184 - loss: 0.2204 - val_accuracy: 0.9168 - val_loss: 0.2311
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 57s 61ms/step - accuracy: 0.9257 - loss: 0.2034 - val_accuracy: 0.9213 - val_loss: 0.2253
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 58s 61ms/step - accuracy: 0.9271 - loss: 0.1964 - 

batch/accuracy,▁▄▄▄▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██▇▇▇▇██▇███████
batch/batch_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇██
batch/learning_rate,█████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁
batch/loss,█▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▄▅▆▆▇▇▇██
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,█▇▆▅▄▃▃▂▂▁
epoch/loss,█▄▃▃▂▂▂▁▁▁
epoch/val_accuracy,▁▂▅▄▄▇▇▆█▇
epoch/val_loss,██▄▅▆▁▁▂▁▂
+3,...
